# Haptic Guidance Experiment Analysis

Loads experiment records and computed metrics from the SQLite database.
Training vs test is determined by scenario name:
- **Training**: scenarios containing `training` (e.g. `training1`)
- **Test**: scenarios containing `Test` (e.g. `Test1`, `Test2`)

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

DB_PATH = Path("web_gui/experiments.db")

METRICS = [
    "path_efficiency",
    "jitter_rms",
    "lateral_error_rms",
    "total_duration",
]

METRIC_LABELS = {
    "path_efficiency":    "Path Efficiency (%)",
    "jitter_rms":         "RMS Jitter (deg)",
    "lateral_error_rms":  "RMS Lateral Error (deg)",
    "total_duration":     "Duration (s)",
}

## Load data

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql("""
        SELECT
            e.id,
            e.run_id,
            e.participant_id,
            e.scenario,
            e.haptic_enabled,
            e.timestamp,
            e.duration,
            m.path_efficiency,
            m.jitter_rms,
            m.jitter_mean,
            m.jitter_std,
            m.lateral_error_rms,
            m.lateral_error_mean,
            m.lateral_error_std,
            m.ideal_path_length,
            m.actual_path_length,
            m.excess_path_length,
            m.total_duration,
            m.total_ticks,
            m.average_tick_rate
        FROM experiments e
        JOIN metrics m ON e.id = m.experiment_id
        WHERE e.status = 'completed'
        ORDER BY e.participant_id, e.scenario, e.timestamp
    """, conn)

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["haptic_enabled"] = df["haptic_enabled"].astype(bool)

# Classify attempt type by scenario name
df["attempt_type"] = df["scenario"].apply(
    lambda s: "test" if "test" in s.lower() else "training"
)

# Attempt number within each participant+scenario group
df["attempt_no"] = df.groupby(["participant_id", "scenario"]).cumcount() + 1

print(f"{len(df)} completed experiments loaded")
df.head()

## Overview

In [ ]:
print("Participants:", sorted(df["participant_id"].unique()))
print("Scenarios:   ", sorted(df["scenario"].unique()))
print()
df.groupby(["participant_id", "scenario", "haptic_enabled"]).size().rename("attempts").reset_index()

## Training progression — metric over attempts

One line per participant per scenario. Set `PARTICIPANT` to `None` to show all.

In [ ]:
PARTICIPANT = None   # e.g. "p1" to focus on one participant
SCENARIO    = None   # e.g. "training1"

train = df[df["attempt_type"] == "training"].copy()
if PARTICIPANT:
    train = train[train["participant_id"] == PARTICIPANT]
if SCENARIO:
    train = train[train["scenario"] == SCENARIO]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Training Progression", fontsize=14, fontweight="bold")

for ax, metric in zip(axes.flat, METRICS):
    for (pid, sc), group in train.groupby(["participant_id", "scenario"]):
        label = f"{pid} / {sc}"
        ax.plot(group["attempt_no"], group[metric], marker="o", label=label)
    ax.set_xlabel("Attempt")
    ax.set_ylabel(METRIC_LABELS[metric])
    ax.set_title(METRIC_LABELS[metric])
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Performance trend — individual lines + group aggregate

Each participant is shown as a thin line. The bold line is the group mean; the shaded band is ±1 std.
Attempt numbers are aligned across participants — only attempts where at least one participant has data are shown.

In [ ]:
METRIC       = "path_efficiency"  # one of METRICS
SCENARIO     = "training1"        # scenario to analyse
ATTEMPT_TYPE = "training"         # "training" or "test"

subset = df[
    (df["attempt_type"] == ATTEMPT_TYPE) &
    (df["scenario"] == SCENARIO)
].copy()

if subset.empty:
    print(f"No data for scenario='{SCENARIO}', attempt_type='{ATTEMPT_TYPE}'")
else:
    # Pivot: rows = attempt_no, columns = participant_id
    pivot = subset.pivot_table(
        index="attempt_no", columns="participant_id",
        values=METRIC, aggfunc="first"
    )

    agg_mean = pivot.mean(axis=1)
    agg_std  = pivot.std(axis=1)

    fig, ax = plt.subplots(figsize=(10, 5))

    # Individual participant lines
    for pid in pivot.columns:
        ax.plot(
            pivot.index, pivot[pid],
            color="steelblue", alpha=0.35, linewidth=1.2,
            marker="o", markersize=4, label=pid
        )

    # Group mean line
    ax.plot(
        agg_mean.index, agg_mean,
        color="navy", linewidth=2.5, marker="o", markersize=6,
        label="Group mean", zorder=5
    )

    # ±1 std band
    ax.fill_between(
        agg_mean.index,
        agg_mean - agg_std,
        agg_mean + agg_std,
        color="navy", alpha=0.12, label="±1 std"
    )

    ax.set_xlabel("Attempt", fontsize=12)
    ax.set_ylabel(METRIC_LABELS[METRIC], fontsize=12)
    ax.set_title(
        f"{METRIC_LABELS[METRIC]} over attempts\n"
        f"scenario: {SCENARIO}  |  n={pivot.shape[1]} participants",
        fontsize=13
    )
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend(fontsize=9, bbox_to_anchor=(1.01, 1), loc="upper left")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## Test results — metric over attempts

Separate plot for test scenarios.

In [ ]:
PARTICIPANT = None
SCENARIO    = None   # e.g. "Test1"

test = df[df["attempt_type"] == "test"].copy()
if PARTICIPANT:
    test = test[test["participant_id"] == PARTICIPANT]
if SCENARIO:
    test = test[test["scenario"] == SCENARIO]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Test Results", fontsize=14, fontweight="bold")

for ax, metric in zip(axes.flat, METRICS):
    for (pid, sc), group in test.groupby(["participant_id", "scenario"]):
        label = f"{pid} / {sc}"
        ax.plot(group["attempt_no"], group[metric], marker="s", label=label)
    ax.set_xlabel("Attempt")
    ax.set_ylabel(METRIC_LABELS[metric])
    ax.set_title(METRIC_LABELS[metric])
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Haptic vs non-haptic comparison

Mean metrics split by haptic enablement, grouped by scenario.

In [ ]:
PARTICIPANT = None

subset = df.copy()
if PARTICIPANT:
    subset = subset[subset["participant_id"] == PARTICIPANT]

summary = (
    subset.groupby(["scenario", "haptic_enabled"])[METRICS]
    .mean()
    .reset_index()
)

scenarios = sorted(summary["scenario"].unique())
x = range(len(scenarios))
width = 0.35

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Haptic vs Non-Haptic (mean per scenario)", fontsize=14, fontweight="bold")

for ax, metric in zip(axes.flat, METRICS):
    for i, haptic in enumerate([True, False]):
        vals = [
            summary.loc[
                (summary["scenario"] == sc) & (summary["haptic_enabled"] == haptic),
                metric
            ].values[0] if not summary.loc[
                (summary["scenario"] == sc) & (summary["haptic_enabled"] == haptic)
            ].empty else 0
            for sc in scenarios
        ]
        offset = (i - 0.5) * width
        ax.bar([xi + offset for xi in x], vals, width, label="Haptic" if haptic else "No Haptic")
    ax.set_xticks(list(x))
    ax.set_xticklabels(scenarios)
    ax.set_ylabel(METRIC_LABELS[metric])
    ax.set_title(METRIC_LABELS[metric])
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## Per-participant summary table

In [ ]:
df.groupby(["participant_id", "scenario", "haptic_enabled"])[METRICS].agg(["mean", "std"]).round(3)

## Raw data export

In [ ]:
# Uncomment to export to CSV
# df.to_csv("experiment_results.csv", index=False)
# print("Exported to experiment_results.csv")